# LiteMLflow quickstart

End-to-end tour of every major feature against a running LiteMLflow server.

**Setup:** start a server (or use the public demo) and set the URL below.

```bash
# Local install
pip install litemlflow
litemlflow serve --addr :5050 --data ./lmf-data &
```

Or use `https://lmf.gorev.space` (the public read-only demo, write-enabled).

What this notebook covers:
1. Create an experiment, group it into a project
2. Log runs with params, metrics, tags, and notes
3. Capture an LLM trace (via the native API or LangChain auto-instrumentation)
4. Register a versioned prompt with aliases
5. Run lineage (parent→child)
6. Register a model, transition stages
7. Compare runs and find them via global search
8. Configure a webhook (echo demo)
9. Build a per-project dashboard via the UI

Each section is self-contained — you can run them in any order after the **Setup** cell.

## 0 · Setup

In [ ]:
TRACKING_URL = "http://localhost:5050"
# TRACKING_URL = "https://lmf.gorev.space"  # public demo

import os, time, random
os.environ["MLFLOW_TRACKING_URI"] = TRACKING_URL

import mlflow
mlflow.set_tracking_uri(TRACKING_URL)
print(f"connected to {TRACKING_URL}")

## 1 · Experiments and projects

Projects are simply experiments tagged with `lmf.project`. Set the tag once and the UI groups experiments accordingly. The dashboard at `#/dashboards/<project>` becomes available immediately.

In [ ]:
exp_name = f"quickstart-rag-{int(time.time())}"
exp_id = mlflow.create_experiment(exp_name, tags={
    "lmf.project": "Quickstart",
    "team": "ml-platform",
    "domain": "rag",
})
print(f"experiment_id={exp_id}")
mlflow.set_experiment(experiment_id=exp_id)

## 2 · Runs with params, metrics, tags

We do a small hyper-parameter sweep so the experiment has multiple comparable runs.

In [ ]:
for k in (3, 5, 8):
    with mlflow.start_run(run_name=f"sweep-k-{k}") as run:
        mlflow.log_param("model", "gpt-4o-mini")
        mlflow.log_param("k", k)
        mlflow.log_param("retriever", "bm25")

        base_recall = 0.55 + 0.04 * k
        for step in range(20):
            mlflow.log_metric("eval/recall@k", base_recall + random.uniform(-0.01, 0.01), step=step)
            mlflow.log_metric("eval/answer_quality", 0.5 + 0.05 * k + random.uniform(-0.01, 0.01), step=step)
            mlflow.log_metric("latency_ms", 350 + k * 30 + random.uniform(-20, 20), step=step)

        mlflow.set_tag("sweep_id", "first-pass")
        # Star the best run so it's easy to spot in the UI.
        if k == 8:
            mlflow.set_tag("lmf.starred", "true")

print("three runs logged — open the experiment and click 'Compare'.")

## 3 · Run notes (markdown)

Notes are markdown-rendered on the run page. Useful for capturing rationale, observations, or things to revisit.

In [ ]:
import requests

# Find the latest run.
res = requests.post(f"{TRACKING_URL}/api/2.0/mlflow/runs/search", json={
    "experiment_ids": [str(exp_id)],
    "max_results": 5,
})
latest_run_id = res.json()["runs"][0]["info"]["run_id"]

note = """# Sweep observation

**Best k=8** — recall@k climbs cleanly with retrieval depth, but latency follows. Next:
- Try a hybrid retriever (BM25 + embedding rerank).
- Cap latency at 600ms, see if quality holds.

Run rationale: pin this branch as `production`.
"""

requests.put(
    f"{TRACKING_URL}/api/v1/runs/{latest_run_id}/note",
    json={"content": note},
).raise_for_status()
print(f"note set on run {latest_run_id}")

## 4 · LLM traces

Two paths: log spans manually via `litemlflow.Client`, or auto-instrument with `litemlflow.langchain.attach()` (requires `pip install 'litemlflow[langchain]'`).

In [ ]:
# Manual trace: pipeline -> retrieve -> generate
from litemlflow import Client

c = Client(TRACKING_URL)
trace_id = c.start_trace()
t0 = time.time_ns()
pipeline = c.log_span(trace_id, "rag.pipeline", run_id=latest_run_id,
                      start_time_ns=t0, end_time_ns=t0 + 800_000_000,
                      attrs={"model": "gpt-4o-mini", "k": 8})
c.log_span(trace_id, "retrieve", run_id=latest_run_id, parent_id=pipeline,
           start_time_ns=t0 + 10_000_000, end_time_ns=t0 + 90_000_000)
c.log_span(trace_id, "generate", run_id=latest_run_id, parent_id=pipeline,
           start_time_ns=t0 + 100_000_000, end_time_ns=t0 + 750_000_000,
           attrs={"prompt_tokens": 320, "completion_tokens": 96})
print(f"trace_id={trace_id} — see 'Trace' tab on run page")

## 5 · Versioned prompts and aliases

Prompts are content-addressed: registering identical content under the same name reuses the version. Aliases (`production`, `candidate`) pin a stable handle while you iterate.

In [ ]:
v1 = c.create_prompt("qs.rag.system", "You are a helpful assistant.", description="v1 baseline")
v2 = c.create_prompt("qs.rag.system", "You are a helpful assistant. Be concise.", description="v2 concise")
v3 = c.create_prompt("qs.rag.system", "You are a helpful assistant. Cite sources by [n].", description="v3 cite")

c.set_prompt_alias("qs.rag.system", "production", v2)
c.set_prompt_alias("qs.rag.system", "candidate",  v3)
print("versions:", v1, v2, v3, "— production points to v2, candidate to v3")

## 6 · Run lineage (parent → child)

Set `mlflow.parentRunId` to nest runs. The UI renders a tree on the run page.

In [ ]:
# Parent run + two children (e.g., a fold-CV pattern).
with mlflow.start_run(run_name="cv-parent") as parent:
    mlflow.log_param("folds", 3)
    parent_id = parent.info.run_id
    for fold in range(3):
        with mlflow.start_run(run_name=f"fold-{fold}", nested=True, tags={"mlflow.parentRunId": parent_id}):
            mlflow.log_param("fold", fold)
            mlflow.log_metric("eval/loss", 0.5 - fold * 0.04)
    mlflow.log_metric("eval/loss", 0.42)  # parent rolled-up
print(f"lineage: parent={parent_id} with 3 nested children")

## 7 · Model registry

Standard MLflow Model Registry endpoints: register a model from a run, transition stages, and resolve by alias.

In [ ]:
model_name = f"qs-rag-{int(time.time())}"
client = mlflow.MlflowClient(TRACKING_URL)
client.create_registered_model(model_name)
mv = client.create_model_version(model_name, source=f"runs:/{latest_run_id}/model", run_id=latest_run_id)
client.transition_model_version_stage(model_name, mv.version, stage="Production")
print(f"registered {model_name} v{mv.version} in Production stage")

## 8 · Compare and search

The UI has dedicated affordances for both, but the API is also straightforward.

In [ ]:
# Native cross-experiment search (returns runs, experiments, prompts).
search = requests.get(f"{TRACKING_URL}/api/v1/search", params={"q": "sweep", "kind": "runs"}).json()
for hit in search.get("items", [])[:5]:
    print(f"{hit['kind']}\t{hit['name']}\t{hit['url']}")

# UI compare URL: open in a browser to see the side-by-side view.
compare_url = f"{TRACKING_URL}/ui/#/experiments/{exp_id}/compare"
print(f"compare: {compare_url}")

## 9 · Webhooks (echo demo)

The built-in `lmf://echo` URL records deliveries to an in-process ring buffer — perfect for verifying the wiring without a public receiver. The ring is visible on the **Webhooks** UI page.

In [ ]:
# Create a demo webhook fired on every run state transition.
wh = requests.post(f"{TRACKING_URL}/api/v1/webhooks", json={
    "name": "quickstart-demo",
    "url": "lmf://echo",
    "events": "run_started,run_finished,run_failed,run_killed",
    "enabled": True,
}).json()
wh_id = wh["id"]
print(f"webhook id={wh_id}")

# Send a test event so something lands in the ring buffer immediately.
requests.post(f"{TRACKING_URL}/api/v1/webhooks/{wh_id}/test", json={}).raise_for_status()

# Read the ring buffer.
echo = requests.get(f"{TRACKING_URL}/api/v1/webhooks/echo?max=5").json()
for e in echo.get("entries", [])[:3]:
    print(f"  {e['event']}\twebhook={e['webhook_id']}\t{e['body'][:80]}")

## 10 · Per-project dashboards

Dashboards are configured via the UI — navigate to `#/dashboards/Quickstart` (or whatever project you set in step 1) and click **Edit** to add widgets:

- **Run count tile** — total / finished / running / failed in the project.
- **Latest best run** — best run by a chosen metric.
- **Run leaderboard** — top N runs sorted by metric.
- **Metric trend chart** — inline SVG chart of one metric across all project runs.

The widget config is stored server-side per (workspace, project), so the same board renders for everyone in the workspace.

## What you should see in the UI

1. `#/experiments` — your experiment under the **Quickstart** project chip.
2. `#/experiments/<id>` — list of 4 runs (3 sweep + 1 with note); toggle **Timeline** to see the Gantt view.
3. Click a run — you'll see the markdown note, the trace tree, and the lineage section.
4. `#/prompts` — `qs.rag.system` with 3 versions and 2 aliases; click → version diff.
5. `#/webhooks` — echo webhook with Recent deliveries panel showing the test event.
6. `#/dashboards/Quickstart` — empty board, click **Edit** to add widgets.
7. Press **⌘K** anywhere — type "sweep", see a unified hit list.